# Tool calling

[Module 04](README.md) · [Workshop home](../README.md) · [Endpoint reference](../docs/inference.md)

**20 minutes.** How a model asks your code to do something, and how you answer.

The model never runs anything. It returns a *request* to run a function. Your code
decides whether to honour it, runs it, and sends the result back. Everything people
call an "agent" is that exchange in a loop.

No framework here. The loop is about fifteen lines and you will write it, because a
loop you wrote is a loop you can debug.

| Step | What |
|---|---|
| 2 | Describe a function to the model |
| 3 | Watch it ask for the function |
| 4 | Send the result back and get an answer |
| 5 | Wrap it in a loop |
| 6 | Turn reasoning on inside the loop |
| 7 | Force or forbid tool use with `tool_choice` |
| 8 | Two calls in one turn |
| 9 | What happens when a tool fails |

## Step 1: Connect

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

root = Path.cwd()
if not (root / ".env").is_file() and (root.parent / ".env").is_file():
    root = root.parent
if not (root / ".env").is_file():
    raise RuntimeError("No .env found. See docs/00_prework.md")

load_dotenv(root / ".env", override=True)
client = OpenAI(
    base_url=os.environ["AKAMAI_BASE_URL"].strip().rstrip("/"),
    api_key=os.environ["AKAMAI_API_KEY"].strip(),
    timeout=120.0,
)

# qwen3-8b for every tool exercise. gemma-4-26b-a4b-it fails at this, see the
# README in this folder for the numbers.
MODEL = "qwen3-8b"
ROOT_DIR = root
print("Ready.")

## Step 2: Describe a function

A tool is a JSON description of a function you are willing to run. The model sees the
name, the description, and the parameters, so all three are prompt text as far as it
is concerned. Vague descriptions produce wrong calls.

Here is a fake fleet database. The data is deliberately not guessable, so you can
tell whether the model really called the function or invented an answer.

In [ ]:
SERVERS = {
    "web-01": {"region": "us-sea", "status": "running", "cpu_percent": 12},
    "web-02": {"region": "us-sea", "status": "running", "cpu_percent": 87},
    "db-01":  {"region": "eu-fra", "status": "stopped", "cpu_percent": 0},
}

def get_server(name: str) -> dict:
    if name not in SERVERS:
        return {"error": f"no server named {name!r}", "known": sorted(SERVERS)}
    return {"name": name, **SERVERS[name]}

TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_server",
        "description": "Look up the live status of one server by its exact name.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Server name, for example web-01"}
            },
            "required": ["name"],
        },
    },
}]

print(get_server("web-02"))
print(get_server("nope"))

## Step 3: The model asks for it

Pass `tools` and the model can respond in one of two ways: a normal text answer, or a
request to call a function. Check `message.tool_calls` to tell them apart.

In [ ]:
messages = [{"role": "user", "content": "Is web-02 under heavy load right now?"}]

first = client.chat.completions.create(
    model=MODEL, messages=messages, tools=TOOLS, max_tokens=300, temperature=0.0
)
msg = first.choices[0].message

print("finish_reason:", first.choices[0].finish_reason)
print("content      :", repr(msg.content))
print()
for call in msg.tool_calls or []:
    print("tool_call_id :", call.id)
    print("function     :", call.function.name)
    print("arguments    :", call.function.arguments, type(call.function.arguments).__name__)

Three things to notice.

`finish_reason` is `tool_calls` and `content` is empty. That is correct behaviour, not
a failure. A model that wants a tool has nothing to say yet.

`arguments` is a **string** containing JSON, not a dictionary. You have to
`json.loads` it, and you should treat it as untrusted input. The model picks those
values.

Every call carries a `tool_call_id`. You must send it back with the result so the
model knows which answer belongs to which question.

## Step 4: Send the result back

Three things go onto the conversation: the assistant's message exactly as it came
back, then one `tool` message per call, each carrying the matching `tool_call_id`.
Then you ask again.

In [ ]:
messages.append(msg)

for call in msg.tool_calls:
    args = json.loads(call.function.arguments)
    result = get_server(**args)
    print(f"ran {call.function.name}({args}) -> {result}")
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps(result),
    })

second = client.chat.completions.create(
    model=MODEL, messages=messages, tools=TOOLS, max_tokens=300, temperature=0.0
)
print("\nfinish_reason:", second.choices[0].finish_reason)
print(second.choices[0].message.content)

The answer should mention 87 percent, which the model could not have known. That is
the whole point of the exercise: a number from your system, in the model's answer.

`tool` message content must be a string, which is why the result goes through
`json.dumps`. Structured JSON reads better to the model than a prose sentence.

## Step 5: The loop

One round trip is rarely enough. A real question might need several lookups, where
each result decides the next one. The loop is: ask, run whatever it wants, ask again,
stop when it answers with no tool calls.

The turn cap is not optional. A model that gets confused will ask for the same thing
forever, and without a cap you will sit there paying for it.

In [ ]:
import textwrap

AVAILABLE = {"get_server": get_server}

def run_tools(question, tools=TOOLS, functions=AVAILABLE, max_turns=6,
              max_tokens=500, think=False, verbose=True):
    messages = [{"role": "user", "content": question}]
    extra = {"extra_body": {"chat_template_kwargs": {"enable_thinking": True}}} if think else {}
    spent = 0

    for turn in range(1, max_turns + 1):
        reply = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            max_tokens=max_tokens, temperature=0.0, **extra
        )
        msg = reply.choices[0].message
        spent += reply.usage.completion_tokens

        # Reasoning comes back in its own field, separate from content, and it is
        # easy to discard by accident. Print all of it.
        reasoning = (getattr(msg, "reasoning", None) or "").strip()
        if verbose:
            print(f"turn {turn}" + (f"  thinking ({len(reasoning)} chars)" if reasoning else ""))
            if reasoning:
                for line in textwrap.wrap(" ".join(reasoning.split()), width=62):
                    print(f"        | {line}")

        if not msg.tool_calls:
            if verbose:
                print(f"        answered  [{spent} output tokens total]\n")
            return msg.content, messages

        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            fn = functions.get(call.function.name)
            result = fn(**args) if fn else {"error": f"no such function {call.function.name!r}"}
            if verbose:
                print(f"        {call.function.name}({args}) -> {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })
        if verbose:
            print()

    return f"gave up after {max_turns} turns", messages

QUESTION = "Compare the CPU load on web-01 and web-02. Which is busier?"
plain, _ = run_tools(QUESTION)
print(plain.strip())

That is the entire mechanism behind every agent framework you have heard of. The rest
is ergonomics.

Notice the model worked out that it needed two lookups and did them without being
told. You described the tool; it decided when to use it.

## Step 6: Reasoning inside the loop

[Module 03](../03_Platform_Features/) turned reasoning on for a single question.
The obvious next thought is to turn it on for an agent, so the model can plan which
tools to use. `run_tools` already takes a `think` flag, so try it.

In [ ]:
thought, _ = run_tools(QUESTION, think=True)
print(thought.strip())

It works. Same two turns, the same two lookups, the same conclusion.

The difference is everything printed under the `|` marks. Read turn 1: the model
works out that it has a lookup tool, that it needs two calls, and which servers to
ask about, all before making a single call. Then read turn 2, where it compares the
two numbers it got back.

Two things are worth noticing in that trace.

**It thinks on every turn, including turn 1**, where the only output that survives is
a tool call. That thinking is written, billed, and then thrown away by the loop.

**Compare the token totals from the two runs.** Roughly five times as many output
tokens for an identical answer. That is the trade. Reasoning earns its cost when
choosing *which* tool to call is genuinely hard. On this question, where the tool
choice is obvious, it buys nothing.

### The part that will bite you

Reasoning is written **before** anything else and spends the same `max_tokens`. In a
loop, that means the model can spend its whole allowance thinking and never get as far
as emitting the tool call.

Watch what one turn does at three different budgets.

In [ ]:
for budget in (120, 200, 300):
    probe = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": QUESTION}],
        tools=TOOLS,
        max_tokens=budget,
        temperature=0.0,
        extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    )
    probe_msg = probe.choices[0].message
    reasoning = getattr(probe_msg, "reasoning", None) or ""
    called = [c.function.name for c in (probe_msg.tool_calls or [])]
    print(f"max_tokens={budget}")
    print(f"   finish_reason : {probe.choices[0].finish_reason}")
    print(f"   reasoning     : {len(reasoning)} chars")
    print(f"   tool_calls    : {called or None}")
    print(f"   content       : {(probe_msg.content or '')[:60]!r}")
    print()

Only the last of the three works.

At 120 and 200 tokens the budget goes on reasoning and runs out before the tool call
is finished. Both come back with `finish_reason: length` and **no `tool_calls`**.

Now look again at the loop. It checks `if not msg.tool_calls` and returns
`msg.content`. Neither of those turns produced a tool call, so the loop treats them as
the final answer and hands back whatever is in `content`. Usually that is an empty
string. Sometimes the model got far enough to start writing the call before it was
cut off, and you get a fragment of its internal syntax, something like
`<tool_call>{"name": "get_server", ...`, as literal text. Either way nothing raises,
and either way it is not an answer.

At 300 tokens there is room for the thinking and the call, and it behaves.

The guard is the `finish_reason` rule from [module 01](../01_Inference_APIs/), applied
inside the loop. `length` never means "finished".

In [ ]:
def run_tools_safe(question, **kwargs):
    """The same loop, refusing to treat a truncated turn as an answer."""
    messages = [{"role": "user", "content": question}]
    extra = {"extra_body": {"chat_template_kwargs": {"enable_thinking": True}}} \
        if kwargs.get("think") else {}
    max_tokens = kwargs.get("max_tokens", 500)

    for turn in range(1, kwargs.get("max_turns", 6) + 1):
        reply = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS,
            max_tokens=max_tokens, temperature=0.0, **extra
        )
        msg = reply.choices[0].message

        # The guard. A truncated turn is an error, not a result.
        if reply.choices[0].finish_reason == "length":
            raise RuntimeError(
                f"turn {turn} hit max_tokens={max_tokens} before finishing. "
                f"Raise the budget, or turn reasoning off."
            )

        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(AVAILABLE[call.function.name](**args)),
            })
    raise RuntimeError("turn cap reached")

try:
    run_tools_safe(QUESTION, think=True, max_tokens=120)
except RuntimeError as exc:
    print("caught:", exc)

print("\nwith room:", run_tools_safe(QUESTION, think=True, max_tokens=800)[:90])

A loud failure instead of an empty string. That is the version to copy into your own
code.

Two rules to carry forward:

- Turning reasoning on means raising `max_tokens` substantially, and in a loop you pay
  it on every turn, not once.
- Treat `finish_reason: "length"` as an error anywhere in an agent loop. A truncated
  turn is not a result, and the default behaviour of a simple loop is to silently
  accept it.

## Step 7: tool_choice

By default the model decides whether to call anything. `tool_choice` takes that
decision away from it.

| Value | Effect |
|---|---|
| `"auto"` | The model decides. The default |
| `"none"` | No tool calls. Answer with text |
| `"required"` | Must call something |
| `{"type": "function", "function": {"name": "..."}}` | Must call that one |

`required` is more useful than it first looks. It is also a way out of the structured
output problem in the next notebook.

In [ ]:
QUESTION = "What is the status of db-01?"

for label, choice in [
    ("auto", "auto"),
    ("none", "none"),
    ("required", "required"),
    ("named", {"type": "function", "function": {"name": "get_server"}}),
]:
    reply = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": QUESTION}],
        tools=TOOLS,
        tool_choice=choice,
        max_tokens=200,
        temperature=0.0,
    )
    m = reply.choices[0].message
    called = [c.function.name for c in (m.tool_calls or [])]
    print(f"{label:<9} finish={reply.choices[0].finish_reason:<11} called={called or 'nothing'}")
    if not called:
        print(f"{'':<9} said: {(m.content or '')[:90]!r}")

Two results there are worth more than the table.

**`none` does not behave well on this endpoint.** Instead of answering in prose, the
model leaks the tool call it wanted to make as literal text in `content`, something
like `<tool_call>{"name": "get_server", ...}</tool_call>`. That is the model's
internal syntax escaping into the answer. Do not rely on `tool_choice: "none"` here.
If you do not want tools used, leave `tools` out of the request entirely.

**Named selection returns `finish_reason: "stop"`** while still carrying a tool call.
So a branch like `if finish_reason == "tool_calls"` will miss it.

Check `msg.tool_calls` rather than `finish_reason` to decide whether to run something.
`finish_reason` is for diagnostics; `tool_calls` is the control flow. The loop in
Step 5 already does it that way.

## Step 8: Two calls in one turn

A model can ask for several things at once when they do not depend on each other.
Each gets its own `tool_call_id`, and you must return a `tool` message for every one.
Returning fewer breaks the next turn.

In [ ]:
reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Give me the status of web-01, web-02 and db-01."}],
    tools=TOOLS,
    max_tokens=400,
    temperature=0.0,
)
calls = reply.choices[0].message.tool_calls or []
print(f"{len(calls)} call(s) in one turn:")
for c in calls:
    print(f"   {c.function.name}({c.function.arguments})")

## Step 9: When a tool fails

The interesting case, and the one most tutorials skip.

`get_server` returns a structured error for an unknown name rather than raising. Watch
what the model does with it.

In [ ]:
answer, history = run_tools("What is the status of web-99?", max_turns=4)
print()
print("answer:", answer)

tool_turns = [m for m in history if isinstance(m, dict) and m.get("role") == "tool"]
print(f"\ntool messages exchanged: {len(tool_turns)}")

`qwen3-8b` handles this one well. It reports the failure instead of inventing a
server, and because the error carried a `known` list, it passes those names on to the
user. Compare that to an error of just `{"error": "not found"}`, which still produces
a sensible reply but a less useful one. What you put in the error shows up in the
answer.

Do not read that as "the model will handle failure." It handled *this* failure, where
there is nothing to retry and the right move is to report it. A tool that fails in a
way the model is supposed to recover from, by correcting its arguments and trying
again, is much less reliable, and that is worth testing yourself with your own tools
rather than assuming either way.

Four things hold regardless:

- Validate arguments before executing anything. The model chose those values, and it
  is a remote caller you do not control.
- Return errors as structured data that says what valid input looks like. The `known`
  list is why the answer above is useful.
- Cap the turns. Always.
- Put retries and fallbacks in your code. A prompt is not an error handler.

A tool is an API you are exposing to an unpredictable caller. Write it that way.

## Validation

In [ ]:
assert first.choices[0].finish_reason == "tool_calls", "the model should have asked for a tool"
assert msg.tool_calls, "no tool_calls on the first reply"
assert "87" in (second.choices[0].message.content or ""), \
    "the final answer should quote 87, the value only the tool knew"

# The loop reaches the same answer with reasoning on and off.
assert plain and "87" in plain, "the plain loop did not report web-02 at 87 percent"
assert thought and "87" in thought, "the reasoning loop did not reach the same answer"

# The guarded loop refuses a truncated turn instead of returning empty content.
try:
    run_tools_safe(QUESTION, think=True, max_tokens=120)
    raise AssertionError("a 120 token budget with reasoning on should have been caught")
except RuntimeError:
    pass

assert len(calls) >= 2, f"expected several calls in one turn, got {len(calls)}"
print("Tool calling complete.")
print("You can describe a tool, run the loop with or without reasoning,")
print("control it with tool_choice, and handle both failing tools and truncated turns.")

## Continue

Next: **[2_build_the_agent.ipynb](2_build_the_agent.ipynb)**, which gives this loop a
real tool, the document search from module 02, and a typed answer at the end.